# 30. Training a Small Transformer End to EndFrom *Transformers from Scratch*.This one trains a model. It takes about a minute and a half.

In [ ]:
# On Colab there is nothing on disk, so this fetches the two or# three files the chapter uses. If you cloned the repository it# finds them where they already are and downloads nothing.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/code/"PLACES = [".", "code", "../code"]for name in ["ch30_corpus.py", "ch30_train.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

A corpus whose rules are known, so learning can be checked.Training on found text makes it hard to say whether a model learned anything, because nobody can state what the right answer was. This corpus is generated from a small grammar written out below, so the structure the model has to discover is known exactly, and generated text can be checked against it rather than admired.

In [ ]:
import randomDETERMINERS = ["the", "a", "every"]ADJECTIVES = ["small", "grey", "hungry", "quiet", "old"]NOUNS = ["cat", "dog", "bird", "fox", "child", "farmer"]VERBS = ["watched", "chased", "found", "carried", "followed"]PLACES = ["garden", "river", "field", "kitchen", "hill"]

`a` becomes `an` before a vowel. A rule the model has to learn.

In [ ]:
def agree(determiner, following):    if determiner == "a" and following[0] in "aeiou":        return "an"    return determiner

determiner, an optional adjective, then a noun.

In [ ]:
def noun_phrase(rng):    words = []    if rng.random() < 0.5:        words.append(rng.choice(ADJECTIVES))    words.append(rng.choice(NOUNS))    return [agree(rng.choice(DETERMINERS), words[0])] + words

noun phrase, verb, noun phrase, and sometimes a place.

In [ ]:
def sentence(rng):    parts = noun_phrase(rng) + [rng.choice(VERBS)] + noun_phrase(rng)    if rng.random() < 0.4:        place = rng.choice(PLACES)        parts += ["in", agree(rng.choice(DETERMINERS), place), place]    return " ".join(parts) + "."

In [ ]:
def build(count=1200, seed=0):    rng = random.Random(seed)    return "\n".join(sentence(rng) for _ in range(count)) + "\n"

Check one line against the grammar that produced the data.

In [ ]:
def is_grammatical(line):    line = line.strip()    if not line.endswith("."):        return False    words = line[:-1].split()    i = 0    def phrase():        nonlocal i        if i >= len(words) or words[i] not in DETERMINERS + ["an"]:            return False        determiner = words[i]        i += 1        if i >= len(words):            return False        mismatch = agree("a", words[i]) != determiner        if determiner in ("a", "an") and mismatch:            return False        if words[i] in ADJECTIVES:            i += 1        if i >= len(words) or words[i] not in NOUNS:            return False        i += 1        return True    if not phrase():        return False    if i >= len(words) or words[i] not in VERBS:        return False    i += 1    if not phrase():        return False    if i < len(words):        if words[i] != "in":            return False        i += 1        if i >= len(words) or words[i] not in DETERMINERS + ["an"]:            return False        determiner = words[i]        i += 1        if i >= len(words) or words[i] not in PLACES:            return False        mismatch = agree("a", words[i]) != determiner        if determiner in ("a", "an") and mismatch:            return False        i += 1    return i == len(words)

In [ ]:
def main():    text = build()    lines = text.strip().split("\n")    print("A CORPUS GENERATED FROM A GRAMMAR")    print(f"  {len(lines)} sentences, {len(text):,} characters")    print(f"  {len(set(text))} distinct characters")    print("  first four sentences")    for line in lines[:4]:        print(f"    {line}")    print("\nEVERY SENTENCE IN IT PASSES THE CHECKER, BY CONSTRUCTION")    passed = sum(is_grammatical(line) for line in lines)    print(f"  {passed} of {len(lines)}")    print("\nAND THE CHECKER REJECTS WHAT IT SHOULD")    for bad in ["cat the watched dog.", "the cat watched.",                "the cat watched the dog in.", "the small grey cat ran.",                "a old farmer chased a bird."]:        print(f"  {bad:<32} {is_grammatical(bad)}")

In [ ]:
main()

Now the second half of the chapter.Training the model, and watching it learn.The architecture is chapter 29's with a batch axis added. Everything else here is the training loop: batches, a loss, an optimizer, a schedule, and a measurement that says whether the model learned the thing it was supposed to learn.

In [ ]:
import jsonimport mathimport osimport timeimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom ch30_corpus import build, is_grammaticalCONTEXT = 64D_MODEL = 96N_HEADS = 4N_LAYERS = 3D_FF = 384BATCH = 32STEPS = 1200WARMUP = 100PEAK_RATE = 3e-3HERE = os.path.dirname(os.path.abspath(__file__))RESULTS = os.path.join(HERE, "ch30_results.json")

In [ ]:
class Block(nn.Module):    def __init__(self):        super().__init__()        self.ln1 = nn.LayerNorm(D_MODEL)        self.qkv = nn.Linear(D_MODEL, 3 * D_MODEL, bias=False)        self.out = nn.Linear(D_MODEL, D_MODEL, bias=False)        self.ln2 = nn.LayerNorm(D_MODEL)        self.up = nn.Linear(D_MODEL, D_FF)        self.down = nn.Linear(D_FF, D_MODEL)    def forward(self, x):        b, n, _ = x.shape        q, k, v = self.qkv(self.ln1(x)).split(D_MODEL, dim=-1)        shape = (b, n, N_HEADS, D_MODEL // N_HEADS)        q, k, v = (t.view(shape).transpose(1, 2) for t in (q, k, v))        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)        x = x + self.out(a.transpose(1, 2).reshape(b, n, D_MODEL))        return x + self.down(F.gelu(self.up(self.ln2(x)),                                    approximate="tanh"))

In [ ]:
class Model(nn.Module):    def __init__(self, vocab):        super().__init__()        self.embed = nn.Embedding(vocab, D_MODEL)        self.pos = nn.Embedding(CONTEXT, D_MODEL)        self.blocks = nn.ModuleList(Block() for _ in range(N_LAYERS))        self.final = nn.LayerNorm(D_MODEL)        self.apply(self._scale)    @staticmethod    def _scale(module):        """Small initial weights. Chapter 7 says why the scale matters;        without this the first loss is twenty times chance."""        if isinstance(module, (nn.Linear, nn.Embedding)):            nn.init.normal_(module.weight, mean=0.0, std=0.02)        if isinstance(module, nn.Linear) and module.bias is not None:            nn.init.zeros_(module.bias)    def forward(self, idx):        n = idx.shape[1]        x = self.embed(idx) + self.pos(torch.arange(n, device=idx.device))        for block in self.blocks:            x = block(x)        return self.final(x) @ self.embed.weight.T

In [ ]:
def batches(data, rng, size=BATCH):    start = torch.randint(len(data) - CONTEXT - 1, (size,),                          generator=rng)    x = torch.stack([data[i:i + CONTEXT] for i in start])    y = torch.stack([data[i + 1:i + CONTEXT + 1] for i in start])    return x, y

In [ ]:
def loss_on(model, data, rng, rounds=8):    model.eval()    total = 0.0    with torch.no_grad():        for _ in range(rounds):            x, y = batches(data, rng)            logits = model(x)            total += float(F.cross_entropy(                logits.reshape(-1, logits.shape[-1]), y.reshape(-1)))    model.train()    return total / rounds

In [ ]:
def rate_at(step):    if step < WARMUP:        return PEAK_RATE * step / WARMUP    progress = (step - WARMUP) / (STEPS - WARMUP)    return PEAK_RATE * 0.5 * (1 + math.cos(math.pi * progress))

In [ ]:
def generate(model, encode, decode, prompt="the ", count=240, seed=0):    rng = torch.Generator().manual_seed(seed)    idx = torch.tensor([encode(prompt)])    for _ in range(count):        logits = model(idx[:, -CONTEXT:])[0, -1]        probs = torch.softmax(logits, dim=-1)        nxt = torch.multinomial(probs, 1, generator=rng)        idx = torch.cat([idx, nxt.view(1, 1)], dim=1)    return decode(idx[0].tolist())

One line, so it fits an output block.

In [ ]:
def flatten(text):    return " / ".join(part.strip() for part in text.split("\n")                      if part.strip())

In [ ]:
def grammatical_share(text):    lines = [line for line in text.split(".")[1:-1]]    lines = [line.strip() + "." for line in lines if line.strip()]    if not lines:        return 0.0, 0    return sum(is_grammatical(line) for line in lines) / len(lines), \        len(lines)

In [ ]:
def train():    torch.manual_seed(0)    text = build()    chars = sorted(set(text))    stoi = {c: i for i, c in enumerate(chars)}    encode = lambda s: [stoi[c] for c in s]    decode = lambda ids: "".join(chars[i] for i in ids)    data = torch.tensor(encode(text), dtype=torch.long)    split = int(0.9 * len(data))    train_data, val_data = data[:split], data[split:]    model = Model(len(chars))    opt = torch.optim.AdamW(model.parameters(), lr=PEAK_RATE,                            weight_decay=0.01)    rng = torch.Generator().manual_seed(1)    eval_rng = torch.Generator().manual_seed(2)    history, samples = [], []    started = time.perf_counter()    for step in range(STEPS + 1):        for group in opt.param_groups:            group["lr"] = rate_at(min(step, STEPS - 1))        if step % 100 == 0:            record = {"step": step,                      "train": loss_on(model, train_data, eval_rng),                      "val": loss_on(model, val_data, eval_rng)}            history.append(record)        if step in (0, 300, STEPS):            text_out = generate(model, encode, decode)            share, count = grammatical_share(text_out)            samples.append({"step": step, "text": flatten(text_out),                            "share": share, "sentences": count})        if step == STEPS:            break        x, y = batches(train_data, rng)        logits = model(x)        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),                               y.reshape(-1))        opt.zero_grad(set_to_none=True)        loss.backward()        opt.step()    return {"history": history, "samples": samples,            "seconds": time.perf_counter() - started,            "parameters": sum(p.numel() for p in model.parameters()),            "vocab": len(chars), "chars": len(text)}

Train once and keep the outcome, so the book builds quickly.

In [ ]:
def results():    if os.path.exists(RESULTS):        return json.load(open(RESULTS))    out = train()    json.dump(out, open(RESULTS, "w"), indent=1)    return out

In [ ]:
def main():    out = results()    print("THE MODEL AND THE DATA")    print(f"  {out['parameters']:,} parameters, vocabulary of "          f"{out['vocab']}")    print(f"  {out['chars']:,} characters, context of {CONTEXT}")    print(f"  {STEPS} steps in {out['seconds']:.0f} seconds on a laptop")    print("\nTHE LOSS, EVERY HUNDRED STEPS")    print("  step    training   held out   gap")    for row in out["history"]:        gap = row["val"] - row["train"]        print(f"  {row['step']:<7} {row['train']:<10.4f} "              f"{row['val']:<10.4f} {gap:+.4f}")    print(f"  chance for {out['vocab']} characters would be "          f"{math.log(out['vocab']):.4f}")    print("\nWHAT IT WRITES, AT THREE POINTS")    for sample in out["samples"]:        if sample["sentences"]:            share = 100 * sample["share"]            count = sample["sentences"]            verdict = f"{share:.0f}% of {count} sentences pass"        else:            verdict = "no complete sentences at all"        print(f"  after {sample['step']} steps, {verdict}")        for start in (0, 68):            piece = sample["text"][start:start + 68]            if piece:                print(f"    {piece}")

In [ ]:
main()

The chapter this came from works through every number above, and explains why each step is there.